In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
import plotly.express as px
from sklearn.metrics import mean_squared_error
from sklearn.utils import shuffle

In [11]:
import seaborn as sns

**week8/training.ipynb** 

In [20]:
autodf = pd.read_csv('data/vehicles.csv')

In [21]:
autodf.columns

Index(['id', 'region', 'price', 'year', 'manufacturer', 'model', 'condition',
       'cylinders', 'fuel', 'odometer', 'title_status', 'transmission', 'VIN',
       'drive', 'size', 'type', 'paint_color', 'state'],
      dtype='str')

In [22]:
autodf.dropna(inplace=True) # cleaning, dropping NaN

In [23]:
autodf = autodf[autodf['price'] > 0] # remove cars with price of zero

In [25]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Drop columns that won't help regression
df = autodf.drop(columns=['id', 'VIN'])  # IDs are meaningless for regression

# --- Ordinal encoding (order matters) ---
ordinal_map = {
    'condition': {'salvage': 0, 'fair': 1, 'good': 2, 'excellent': 3, 'like new': 4, 'new': 5},
    'cylinders': {'3 cylinders': 3, '4 cylinders': 4, '5 cylinders': 5,
                  '6 cylinders': 6, '8 cylinders': 8, '10 cylinders': 10, '12 cylinders': 12},
    'size':      {'sub-compact': 0, 'compact': 1, 'mid-size': 2, 'full-size': 3},
}

for col, mapping in ordinal_map.items():
    df[col] = df[col].map(mapping)

# --- One-hot encoding (no natural order) ---
nominal_cols = ['region', 'manufacturer', 'model', 'fuel',
                'title_status', 'transmission', 'drive',
                'type', 'paint_color', 'state']

df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

# --- Handle nulls ---
df = df.fillna(df.median(numeric_only=True))

# --- Ready for regression ---
X = df.drop(columns=['price'])
y = df['price']

In [26]:
X

,year,condition,cylinders,odometer,size,region_abilene,region_akron / canton,region_albany,region_albuquerque,region_altoona-johnstown,...,state_sd,state_tn,state_tx,state_ut,state_va,state_vt,state_wa,state_wi,state_wv,state_wy
215,2002.0,3,4.0,155000.0,1,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
219,1995.0,1,6.0,110661.0,2,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
268,2008.0,3,4.0,56700.0,1,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
337,2011.0,3,6.0,164000.0,3,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
338,1972.0,1,6.0,88100.0,3,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
426785,2015.0,4,8.0,146795.0,3,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
426788,2016.0,4,4.0,61127.0,1,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
426792,2014.0,3,8.0,154642.0,3,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
426793,2018.0,3,4.0,36465.0,2,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True


In [27]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# --- Scale features (critical for Lasso) ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- LassoCV auto-finds the best alpha ---
lasso = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso.fit(X_scaled, y)

# --- Extract coefficients ---
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': lasso.coef_
}).sort_values('coefficient', key=abs, ascending=False)

# Features Lasso kept (non-zero)
selected = coef_df[coef_df['coefficient'] != 0]
print(f"Best alpha: {lasso.alpha_:.4f}")
print(f"Features kept: {len(selected)} / {len(X.columns)}")
print(selected.head(20))

Best alpha: 21.0791
Features kept: 3162 / 5466
                         feature  coefficient
0                           year  5669.642348
5381                    fuel_gas -2513.519588
5402                  type_truck  2434.454932
3                       odometer -1880.283496
5390          transmission_other -1401.356176
404         manufacturer_ferrari  1396.246685
5400                 type_pickup  1306.775608
1                      condition  1289.312697
5391                   drive_fwd -1288.015087
4712  model_super duty f-550 drw  1269.684798
4300                 model_sedan  1146.034553
5157                 model_wagon  1054.622565
1711              model_corvette  1005.387158
3272                   model_k10   962.462179
1726                 model_coupe   860.571960
5382                 fuel_hybrid  -853.153285
425         manufacturer_porsche   798.378152
5401                  type_sedan  -759.174501
2                      cylinders   755.661832
1317                model_bronco 

In [40]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# --- Scale features (critical for Lasso) ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# --- Optional: find a good alpha manually ---
# Run this block once to pick your alpha, then hardcode it below
alphas = [0.1, 1, 10, 100, 500, 1000]
for a in alphas:
    m = Lasso(alpha=a, max_iter=10000)
    m.fit(X_train, y_train)
    n_kept = np.sum(m.coef_ != 0)
    r2 = r2_score(y_test, m.predict(X_test))
    print(f"alpha={a:>6} | features kept: {n_kept:>3} | R²: {r2:.3f}")

# --- Fit with your chosen alpha ---
alpha = 100  # <-- swap this after reviewing the output above

lasso = Lasso(alpha=alpha, max_iter=10000)
lasso.fit(X_train, y_train)

# --- Extract coefficients ---
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': lasso.coef_
}).sort_values('coefficient', key=abs, ascending=False)

# Features Lasso kept (non-zero)
selected = coef_df[coef_df['coefficient'] != 0]
zeroed   = coef_df[coef_df['coefficient'] == 0]

print(f"\nAlpha: {alpha}")
print(f"Features kept:   {len(selected)} / {len(X.columns)}")
print(f"Features zeroed: {len(zeroed)}")
print(f"R² on test set:  {r2_score(y_test, lasso.predict(X_test)):.3f}")
print("\nTop features:")
print(selected.head(20).to_string(index=False))

/home/fazuskazoo/anaconda3/envs/ucb/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.448e+10, tolerance: 4.541e+08
  model = cd_fast.enet_coordinate_descent(


alpha=   0.1 | features kept: 4883 | R²: 0.641
alpha=     1 | features kept: 4809 | R²: 0.648
alpha=    10 | features kept: 3994 | R²: 0.653
alpha=   100 | features kept: 757 | R²: 0.596
alpha=   500 | features kept:  46 | R²: 0.487
alpha=  1000 | features kept:  16 | R²: 0.432

Alpha: 100
Features kept:   757 / 5466
Features zeroed: 4709
R² on test set:  0.596

Top features:
                   feature  coefficient
                      year  4616.858549
                  fuel_gas -2935.490722
                  odometer -2480.265848
                type_truck  2381.300170
                 drive_fwd -1628.643161
                 condition  1321.340801
                 cylinders  1312.008094
        transmission_other -1295.238634
               type_pickup  1155.897189
               model_sedan  1014.473051
model_super duty f-550 drw   909.918983
               fuel_hybrid  -885.058967
                 model_k10   829.663385
                type_sedan  -747.004489
                     